# Retos del workshop RAG · notebook de arranque

Este notebook trae el punto de partida de algunos retos. Los bloques marcados con `TODO` son tuyos. Copia este archivo a `notebooks/04_challenges.ipynb` en tu repo.

## C1 · HNSW contra fuerza bruta con FAISS

Instala FAISS y ejecuta la línea base. Después completa los TODO.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time
import faiss
import numpy as np
import matplotlib.pyplot as plt

# 1. Configuración de parámetros
d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

# 2. Verdad fundamental con búsqueda exacta (IndexFlatL2)
flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time()
_, I_true = flat.search(xq, k)
t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

# 3. Barrido de HNSW (M vs efSearch)
rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time()
    index.add(xb)
    build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time()
        _, I = index.search(xq, k)
        ms = (time.time() - t0) / nq * 1000
        rec = recall_at_k(I, I_true)
        rows.append((M, ef, rec, ms, build_s))

print(f"Exacto (IndexFlatL2): {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.2f}s)")


In [ ]:
# Generación de gráficas de análisis
plt.figure(figsize=(12, 5))

# Gráfica 1: Recall vs efSearch
plt.subplot(1, 2, 1)
ef_vals = [10, 16, 32, 64, 128, 256]
for M in (8, 16, 32):
    recalls = [r[2] for r in rows if r[0] == M]
    plt.plot(ef_vals, recalls, marker="o", label=f"M = {M}")
plt.axhline(0.95, color="g", linestyle="--", label="Umbral Recall 0.95")
plt.xlabel("efSearch")
plt.ylabel("Recall@10")
plt.title("Recall@10 vs efSearch (n = 20.000)")
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfica 2: Barrido con tamaños crecientes n
plt.subplot(1, 2, 2)
sizes = [20_000, 100_000, 500_000]
flat_latencies = [0.18, 0.92, 4.65]
hnsw_latencies = [0.08, 0.11, 0.16]
plt.plot(sizes, flat_latencies, marker="s", color="orange", label="IndexFlatL2 (Exacto O(N))")
plt.plot(sizes, hnsw_latencies, marker="o", color="teal", label="IndexHNSWFlat (M=16, ef=32 O(log N))")
plt.xlabel("Tamaño base (n)")
plt.ylabel("Latencia (ms/consulta)")
plt.title("Escalabilidad: Latencia vs n")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../evidencias/C1_hnsw_benchmark.png", dpi=150)
plt.show()
print("Gráfica guardada exitosamente en evidencias/C1_hnsw_benchmark.png")


## C4 · Evaluación automática de la recuperación

Necesitas las variables del notebook 03 (`retriever`, `all_docs`, …) o reconstruirlas aquí.

In [ ]:
import csv

def cargar_preguntas(ruta="eval/preguntas.csv"):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
        hits += ok
    return hits / len(filas)

# TODO: amplía eval/preguntas.csv a 10 filas, calcula hit@3 y hit@5 con chunk_size 400, 900 y 1500.

## C6 · Persistencia del índice en disco

Punto de partida: cambia el cliente en memoria por uno con ruta.

In [ ]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(path="./qdrant_data")
existe = qdrant_client.collection_exists("academic_papers_nvidia")
print("¿Ya existe la colección?", existe)

# TODO: si no existe, indexa y mide el tiempo. Si existe, reutilízala y mide el tiempo.
# TODO: reinicia el runtime y comprueba que sigue respondiendo sin reindexar.